In [1]:
"""
--------------------------------------------------------------------------------
Set up GEE API
--------------------------------------------------------------------------------
"""
import ee
import geemap
import geetools
import time
from pprint import pprint
import GEE_utils

In [ ]:
"""
--------------------------------------------------------------------------------
Define parameters for study: CRS, ROI, etc.
--------------------------------------------------------------------------------
"""
PROJ = ee.Projection('EPSG:32647')
SCALE = 1e4                         # pixel size (meters)
SIZE = 46                           # how many pixels per side
SIZE *= SCALE
CM_X, CM_Y = 580000, 2100000

# Define ROI (fixed square region w/ side length SIZE * SCALE)
ROI = ee.Geometry.Rectangle(
    coords=[
        CM_X - SIZE / 2,
        CM_Y - SIZE / 2,
        CM_X + SIZE / 2,
        CM_Y + SIZE / 2
    ],
    proj=PROJ,
    evenOdd=False
)

GRID = ROI.coveringGrid(PROJ, SCALE)

# Initialize Map (for visualization)
Map = geemap.Map()
Map.addLayer(GRID, {}, 'grid')
Map.centerObject(ROI, zoom = 6)

In [25]:
START_DATE = ee.Date('2018-07-04')
END_DATE = ee.Date('2022-12-31')
CHUNK_SIZE = 1

# Initialize Map (for visualization)
Map = geemap.Map()
Map.addLayer(GRID, {}, 'grid')
Map.centerObject(ROI, zoom = 6)

# Import data
pm25_ic = ee.ImageCollection('projects/sat-io/open-datasets/GHAP/GHAP_D1K_PM25')
weather_ic = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
fire_ic = ee.ImageCollection('NASA/VIIRS/002/VNP14A1')
terrain_img = ee.Image('CGIAR/SRTM90_V4')

weather_scale = weather_ic.first().projection().nominalScale()
fire_scale = fire_ic.first().projection().nominalScale()

PROJ = ee.Projection('EPSG:32647')
SIZE = 43                           # how many pixels per side
SIZE *= weather_scale.getInfo()
CM_X, CM_Y = 580000, 2100000

# Define ROI (fixed square region w/ side length SIZE * SCALE)
ROI = ee.Geometry.Rectangle(
    coords=[
        CM_X - SIZE / 2,
        CM_Y - SIZE / 2,
        CM_X + SIZE / 2,
        CM_Y + SIZE / 2
    ],
    proj=PROJ,
    evenOdd=False
)

GRID = ROI.coveringGrid(PROJ, SCALE)

# Initialize Map (for visualization)
Map = geemap.Map()
Map.addLayer(GRID, {}, 'grid')
Map.centerObject(ROI, zoom = 6)

ics = {
    'pm25': pm25_ic.filterBounds(ROI),
    'weather': weather_ic.filterBounds(ROI),
    'fire': fire_ic.filterBounds(ROI)
}

print(weather_scale.getInfo())
num_images = 0
current_start = START_DATE
while True:
    current_end = current_start.advance(CHUNK_SIZE, 'week')
    if current_end.millis().getInfo() > END_DATE.millis().getInfo():
        current_end = END_DATE

    if current_start.millis().getInfo() > END_DATE.millis().getInfo():
        break

    print(
        current_start.format('YYYY-MM-dd').getInfo(),
        current_end.format('YYYY-MM-dd').getInfo(),
        num_images
    )

    dates = GEE_utils.create_date_list(current_start, current_end)

    image = GEE_utils.create_date_image(ics, terrain_img, weather_scale, fire_scale, START_DATE)

    image_list = dates.map(
        lambda date: GEE_utils.create_date_image(ics, terrain_img, weather_scale, fire_scale, date)
    ).filter(ee.Filter.eq('num_bands', 10))

    images = ee.ImageCollection.fromImages(image_list)

    tasks = ee.batch.Export.geetools.imagecollection.toDrive(
        imagecollection = images,
        index_property = 'date',
        description = 'dataset_1',
        scale = weather_scale,
        crs = PROJ.crs(),
        region = ROI,
        folder = 'dataset_1',
    )

    for task in tasks:
        time.sleep(0.01)
        task.start()

    num_images += images.size().getInfo()

    current_start = current_end.advance(1, 'day')


11131.949079327358
2018-07-04 2018-07-11 0
2018-07-12 2018-07-19 8
2018-07-20 2018-07-27 16
2018-07-28 2018-08-04 24
2018-08-05 2018-08-12 32
2018-08-13 2018-08-20 40
2018-08-21 2018-08-28 48
2018-08-29 2018-09-05 56
2018-09-06 2018-09-13 64
2018-09-14 2018-09-21 72
2018-09-22 2018-09-29 80
2018-09-30 2018-10-07 88
2018-10-08 2018-10-15 96
2018-10-16 2018-10-23 104
2018-10-24 2018-10-31 112
2018-11-01 2018-11-08 120
2018-11-09 2018-11-16 128
2018-11-17 2018-11-24 136
2018-11-25 2018-12-02 144
2018-12-03 2018-12-10 152
2018-12-11 2018-12-18 160
2018-12-19 2018-12-26 168
2018-12-27 2019-01-03 176
2019-01-04 2019-01-11 184
2019-01-12 2019-01-19 192
2019-01-20 2019-01-27 199
2019-01-28 2019-02-04 206
2019-02-05 2019-02-12 214
2019-02-13 2019-02-20 222
2019-02-21 2019-02-28 230
2019-03-01 2019-03-08 238
2019-03-09 2019-03-16 246
2019-03-17 2019-03-24 254
2019-03-25 2019-04-01 262
2019-04-02 2019-04-09 270
2019-04-10 2019-04-17 278
2019-04-18 2019-04-25 286
2019-04-26 2019-05-03 294
2019-05-

In [37]:
ee.data.listAssets('projects/ee-thailand-pm/assets')

{'assets': [{'type': 'IMAGE_COLLECTION',
   'name': 'projects/ee-thailand-pm/assets/test_export',
   'id': 'projects/ee-thailand-pm/assets/test_export',
   'updateTime': '2025-04-13T02:40:14.493432Z'}]}

In [35]:
ee.data.deleteAsset('projects/ee-thailand-pm/assets/test_export')